In [1]:
import keras
from keras import layers

import numpy as np
import pandas as pd
import random
import io

In [ ]:
df=next(pd.read_csv("song_lyrics.csv"))[["title","tag","artist","views","lyrics"]]
text=df["title"].str.cat(sep=" ")
print(text)

Killa Cam Can I Live Forgive Me Father Down and Out Fly In Lollipop Remix Im Not You Family Ties Rockin and Rollin Lord You Know Money On My Mind Think Yall Know DEvils December 4th 98 Freestyle What Happened to That Boy? Its Hot Some Like It Hot Losing Weight Pt. 2 Its Like That More Gangsta Music Mr. Carter The Liberation of Lonzo Williams Encore / Curtains Down Foo Dang Love Is Love Sunshine Time Hey AZ Colors Broads Fly Out Im Blooded A Milli Big Girl C.R.E.A.M. Warning Never Change Steady Mobbin’ Suck My Dick Comedy Central Im a Hustla Carter II Ride for My Niggas Sky is the Limit Pop Bottles Harlem Streets Dr. C. Ph.D Barry Bonds Fireman On My Own Whatever I Wan Do Brooklyn Go Hard Juicy D.O.A. Death of Auto-Tune Youre Nobody Til Somebody Kills You Me  My Bitch Major Pain  Bossalini Brooklyn Zoo The What Im Gettin Money Not Give a Fuck Ten Crack Commandments Niggaz Knock Yourself Out Hate We Gonna Make It Back That Azz Up A Star Is Born A Milli I Get Crazy Can I Get A… Already Ho

In [3]:
text = text.replace("\n", " ")  # We remove newlines chars for nicer display
print("Corpus length:", len(text))

chars = sorted(list(set(text)))
print("Total chars:", len(chars))
char_indices = dict((c, i) for i, c in enumerate(chars))
indices_char = dict((i, c) for i, c in enumerate(chars))

# cut the text in semi-redundant sequences of maxlen characters
maxlen = 40
step = 3
sentences = []
next_chars = []
for i in range(0, len(text) - maxlen, step):
    sentences.append(text[i : i + maxlen])
    next_chars.append(text[i + maxlen])
print("Number of sequences:", len(sentences))

x = np.zeros((len(sentences), maxlen, len(chars)), dtype="bool")
y = np.zeros((len(sentences), len(chars)), dtype="bool")
for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        x[i, t, char_indices[char]] = 1
    y[i, char_indices[next_chars[i]]] = 1

Corpus length: 149290
Total chars: 110
Number of sequences: 49750


In [4]:
model = keras.Sequential(
    [
        keras.Input(shape=(maxlen, len(chars))),
        layers.LSTM(128),
        layers.Dense(len(chars), activation="softmax"),
    ]
)
optimizer = keras.optimizers.RMSprop(learning_rate=0.01)
model.compile(loss="categorical_crossentropy", optimizer=optimizer)

In [5]:
def sample(preds, temperature=1.0):
    # helper function to sample an index from a probability array
    preds = np.asarray(preds).astype("float64")
    preds = np.log(preds) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

In [6]:
epochs = 20
batch_size = 128

for epoch in range(epochs):
    model.fit(x, y, batch_size=batch_size, epochs=1)
    print()
    print("Generating text after epoch: %d" % epoch)

    start_index = random.randint(0, len(text) - maxlen - 1)
    for diversity in [0.2, 0.5, 1.0, 1.2]:
        print("...Diversity:", diversity)

        generated = ""
        sentence = text[start_index : start_index + maxlen]
        print('...Generating with seed: "' + sentence + '"')

        for i in range(50):
            x_pred = np.zeros((1, maxlen, len(chars)))
            for t, char in enumerate(sentence):
                x_pred[0, t, char_indices[char]] = 1.0
            preds = model.predict(x_pred, verbose=0)[0]
            next_index = sample(preds, diversity)
            next_char = indices_char[next_index]
            sentence = sentence[1:] + next_char
            generated += next_char

        print("...Generated: ", generated)
        print("-")

389/389 ━━━━━━━━━━━━━━━━━━━━ 11s 27ms/step - loss: 2.6179

Generating text after epoch: 0
...Diversity: 0.2
...Generating with seed: " AM听不懂 Wisdom in Theory Shaolin vs Wu Ta"
...Generated:  me Bemix Soon Shay May The Ball Same Beat Me Say B
-
...Diversity: 0.5
...Generating with seed: " AM听不懂 Wisdom in Theory Shaolin vs Wu Ta"
...Generated:  yk Alle Not Sistin Say Day Cay Sho Rey Now The Sta
-
...Diversity: 1.0
...Generating with seed: " AM听不懂 Wisdom in Theory Shaolin vs Wu Ta"
...Generated:   Lemes At Ramt Smisayl Batco Cay Foo Shome He thea
-
...Diversity: 1.2
...Generating with seed: " AM听不懂 Wisdom in Theory Shaolin vs Wu Ta"
...Generated:  boythew Heanhinackaidy SorevC Sumet Wamfe YrugéAJt
-
389/389 ━━━━━━━━━━━━━━━━━━━━ 11s 27ms/step - loss: 2.2232

Generating text after epoch: 1
...Diversity: 0.2
...Generating with seed: "agic Hood Politics The Breakup Song Atla"
...Generated:  te Stald Stat It You Wat It Got You Bat Back Wat I
-
...Diversity: 0.5
...Generating with seed: "agic H

KeyboardInterrupt: 

In [ ]:
model.save("TitleGen.keras")